# Teste Final do Pipeline Multimodal com LangGraph + AWS S3

Este notebook realiza a validação completa do pipeline multimodal desenvolvido para apoio à triagem clínica preventiva em saúde feminina.

O fluxo integra:
- upload de arquivos multimodais no Amazon S3;
- download automático pelo LangGraph;
- processamento de vídeo;
- processamento de áudio;
- fusão multimodal;
- geração de resposta interpretativa com LLM.

O pipeline utiliza:
- AWS S3;
- AWS Rekognition;
- YOLOv8;
- OpenCV;
- Librosa;
- LangGraph;
- RAG;
- Mistral via Ollama.

Os testes utilizam arquivos da base pública RAVDESS, composta por atores e emoções simuladas, evitando uso de dados reais de pacientes.

A análise possui finalidade exclusivamente educacional e de apoio à triagem preventiva.

In [1]:
import sys
print(sys.executable)

C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\venv\Scripts\python.exe


In [2]:
!{sys.executable} -m pip install moviepy imageio imageio-ffmpeg


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
!{sys.executable} -m pip show moviepy

Name: moviepy
Version: 2.2.1
Summary: Video editing with Python
Home-page: 
Author: Zulko 2024
Author-email: 
License: MIT License
Location: C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\venv\Lib\site-packages
Requires: decorator, imageio, imageio_ffmpeg, numpy, pillow, proglog, python-dotenv
Required-by: 


In [4]:
from moviepy import VideoFileClip

print("MoviePy funcionando.")

MoviePy funcionando.


In [5]:
import sys
sys.path.append("..")

import os
import boto3

from pathlib import Path
from pprint import pprint

from dotenv import load_dotenv

from src.workflows.langgraph_flow import (
    build_medical_assistant_graph
)

from src.llm.ollama_client import (
    get_llm
)

from src.rag.vector_store import (
    load_vector_store
)

from src.rag.retriever import (
    get_retriever,
    retrieve_context
)

from src.multimodal.media_utils import (
    extract_audio_from_video
)

In [6]:
llm = get_llm(
    model_name="mistral",
    temperature=0.2
)

vector_store = load_vector_store(
    "../data/vector_store"
)

retriever = get_retriever(
    vector_store,
    k=3
)

app = build_medical_assistant_graph(
    llm=llm,
    retriever=retriever,
    db_path="../data/medical_demo.db"
)

print("LangGraph compilado com sucesso.")

C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\notebooks\..\src\llm\ollama_client.py:4: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  return ChatOllama(


Base vetorial carregada de: ../data/vector_store
LangGraph compilado com sucesso.


### CONFIGURAÇÃO AWS
Nesta etapa são carregadas as variáveis de ambiente e a configuração do Amazon S3 utilizada para armazenamento temporário dos arquivos multimodais.

In [7]:
load_dotenv()

bucket = os.getenv("AWS_S3_BUCKET")
region = os.getenv("AWS_REGION")

print("Bucket:", bucket)
print("Region:", region)

Bucket: postechfase4
Region: us-east-1


In [8]:
s3 = boto3.client(
    "s3",
    region_name=region
)

print("Cliente S3 criado com sucesso.")

Cliente S3 criado com sucesso.



### ESCOLHA DO CENÁRIO

Nesta etapa é selecionado o cenário multimodal utilizado para teste.

Os vídeos utilizados são provenientes de bancos públicos de mídia e representam situações simuladas de interação emocional. Os áudios utilizados foram criados de forma sintética em português para validar diferentes níveis de risco emocional sem uso de dados reais de pacientes.

Cenários sugeridos:

- baixo_risco
  - expressão neutra ou positiva
  - fala tranquila
  - ausência de palavras de alerta

- risco_medio
  - sinais leves de tristeza, cansaço ou preocupação
  - hesitação na fala
  - desconforto emocional moderado

- risco_alto
  - sinais aparentes de medo ou sofrimento
  - fala com palavras de alerta
  - possível tensão emocional

A combinação entre vídeo e áudio permite validar:
- análise visual;
- análise acústica;
- interpretação textual;
- fusão multimodal;
- geração automática de alertas preventivos.

## Cenário 1 -> Happy

In [9]:
videoHappy = r"C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\video\HappyPregnantWoman.mp4"

print("Vídeo existe?", os.path.exists(videoHappy))
print(videoHappy)

Vídeo existe? True
C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\video\HappyPregnantWoman.mp4


## EXTRAÇÃO DO ÁUDIO

O pipeline possui suporte para extração automática de áudio a partir de arquivos de vídeo. Essa funcionalidade permite processar vídeos contendo fala original, separando automaticamente a trilha sonora para posterior análise acústica e textual.

A extração é realizada utilizando bibliotecas de processamento multimídia integradas ao projeto.

Nesta demonstração, entretanto, a etapa de extração automática não será utilizada, pois:
- os vídeos selecionados para análise visual não possuem áudio original;
- áudios sintéticos em português foram criados separadamente para simular diferentes cenários emocionais;
- essa abordagem permite maior controle sobre os testes multimodais e evita uso de dados reais de pacientes.

Mesmo não sendo utilizada neste cenário específico, a funcionalidade permanece implementada e disponível para futuros testes com vídeos completos contendo áudio integrado.

In [10]:
audio_output_extract = Path(
    r"C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\audio\extract_happy_audio_female.wav"
)

extract_audio_from_video(
    video_path=str(videoHappy),
    output_audio_path=str(audio_output_extract)
)

print("Áudio extraído com sucesso.")

O vídeo não possui faixa de áudio.
Áudio extraído com sucesso.


In [11]:
audio_output = Path(
    r"C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\audio\happy.wav"
)
print("Áudio encontrado?", audio_output.exists())
print(audio_output)

Áudio encontrado? True
C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\audio\happy.wav


### UPLOAD S3

Nesta etapa os arquivos multimodais são enviados para o Amazon S3.

O pipeline LangGraph realizará o download automático durante a execução.

In [12]:
video_s3_key = (
    "inputs/videos/HappyPregnantWoman.mp4"
)

audio_s3_key = (
    "inputs/audios/happy.wav"
)

s3.upload_file(
    str(videoHappy),
    bucket,
    video_s3_key
)

print("Vídeo enviado para o S3.")

Vídeo enviado para o S3.


In [13]:
s3.upload_file(
    str(audio_output),
    bucket,
    audio_s3_key
)

print("Áudio enviado para o S3.")

Áudio enviado para o S3.


In [14]:
for key in [video_s3_key, audio_s3_key]:

    response = s3.head_object(
        Bucket=bucket,
        Key=key
    )

    print(
        key,
        "-",
        response["ContentLength"],
        "bytes"
    )

inputs/videos/HappyPregnantWoman.mp4 - 12282853 bytes
inputs/audios/happy.wav - 1097694 bytes


### BUILD LANGGRAPH

Nesta etapa o LangGraph é compilado para execução do pipeline multimodal integrado.

In [15]:
app = build_medical_assistant_graph(
    llm=llm,
    retriever=retriever
)

print("LangGraph compilado com sucesso.")

LangGraph compilado com sucesso.


### CRIAÇÃO DO ESTADO

O estado contém:
- pergunta do usuário;
- chaves dos arquivos multimodais no S3;
- idioma do áudio;
- identificação do cenário.

In [16]:
state = {

    "question": (
        "Avalie possíveis sinais de risco "
        "emocional na análise multimodal."
    ),

    "audio_s3_key": audio_s3_key,

    "video_s3_key": video_s3_key,

    "audio_language": "pt-BR",

    "patient_id": "happy_woman1"
}

### EXECUÇÃO DO PIPELINE

O pipeline executa automaticamente:
- download dos arquivos do S3;
- análise de vídeo;
- análise de áudio;
- fusão multimodal;
- geração de resposta;
- alerta preventivo.

In [17]:
result = app.invoke(state)

print("Pipeline executado com sucesso.")

Pipeline executado com sucesso.


### RESULTADO DO VÍDEO

Nesta etapa é exibido o resultado da análise visual.

In [18]:
pprint(result["video_result"])

{'flags': ['person_detected', 'face_detected', 'possible_body_tension'],
 'interpretation': ['Foram detectadas faces no vídeo, permitindo análise de '
                    'expressões emocionais aparentes.',
                    'A distribuição percentual das emoções aparentes '
                    'consideradas na análise foi: HAPPY: 100.0%.',
                    'Foram observados possíveis sinais posturais persistentes '
                    'de tensão corporal, tratados apenas como evidência '
                    'complementar e exploratória.',
                    'A análise facial não identificou sinais visuais '
                    'relevantes de desconforto emocional.'],
 'limitations': ['A análise facial indica apenas expressões aparentes, não '
                 'estado psicológico real.',
                 'O sistema não realiza diagnóstico médico ou psicológico.',
                 'Os sinais visuais devem ser interpretados apenas como apoio '
                 'à triagem.',
       

In [19]:
for item in result["video_result"].get(
    "interpretation",
    []
):
    print("-", item)

- Foram detectadas faces no vídeo, permitindo análise de expressões emocionais aparentes.
- A distribuição percentual das emoções aparentes consideradas na análise foi: HAPPY: 100.0%.
- Foram observados possíveis sinais posturais persistentes de tensão corporal, tratados apenas como evidência complementar e exploratória.
- A análise facial não identificou sinais visuais relevantes de desconforto emocional.


In [20]:
from src.multimodal.video_processor import analyze_video

video_result = analyze_video(videoHappy)

print("Posture score:", video_result.get("posture_score"))
print("Posture flags:", video_result.get("posture_flags"))
print("Frames com pose:", video_result.get("metadata", {}).get("frames_with_pose"))

video_result.get("posture_interpretation")

Posture score: 0.0
Posture flags: ['possible_body_tension']
Frames com pose: 20


['Foram observados possíveis sinais posturais persistentes de tensão corporal, tratados apenas como evidência complementar e exploratória.']

## Interpretação da Análise de Postura Corporal — Cenário de Baixo Risco

Nesta etapa, o pipeline executa a análise complementar de postura corporal utilizando YOLOv8 Pose em um vídeo de teste associado a um cenário emocional positivo ou neutro.

O resultado apresentou:
- `posture_score = 0.0`;
- detecção da flag `possible_body_tension`;
- presença de pose identificada em 20 frames analisados.

Apesar da identificação de possíveis sinais de tensão corporal, a análise visual geral permaneceu classificada como **baixo risco**, uma vez que:
- não foram detectadas emoções negativas persistentes;
- não houve predominância de medo ou tristeza;
- as expressões faciais permaneceram compatíveis com um cenário emocional estável.

A flag `possible_body_tension` pode surgir naturalmente em vídeos contendo:
- movimentação corporal;
- mudanças de postura;
- gesticulação durante a fala;
- inclinação do tronco;
- desalinhamento momentâneo dos ombros;
- limitações do enquadramento da câmera.

Por esse motivo, os sinais corporais recebem:
- baixa ponderação no score final;
- interpretação conservadora;
- uso exclusivamente complementar.

A interpretação gerada pelo sistema reforça esse comportamento ao indicar que:
- os sinais corporais foram tratados apenas como apoio contextual;
- não houve evidência visual relevante de desconforto emocional.


### RESULTADO DO ÁUDIO

Nesta etapa é exibido o resultado da análise acústica e textual do áudio.

In [21]:
pprint(result["audio_result"])

{'detected_categories': {},
 'flags': ['voice_instability', 'elevated_voice_tension', 'speech_hesitation'],
 'interpretation': 'O perfil vocal estimado é compatível com voz feminina. Não '
                   'foram identificadas palavras-chave clínicas de alerta na '
                   'transcrição. Os sinais observados vieram principalmente de '
                   'características acústicas da voz, como variação de pitch, '
                   'pausas, energia ou intensidade vocal. Como a transcrição '
                   'não contém termos clínicos de alerta, esses sinais devem '
                   'ser tratados como evidências complementares de baixa '
                   'confiança. A intensidade vocal foi classificada como alta. '
                   'Foram registradas oscilações vocais, que podem indicar '
                   'variação emocional, mas isoladamente não confirmam '
                   'ansiedade ou tensão. Foram identificadas pausas ou '
                   'hesitações na 

In [22]:
print(
    result["audio_result"].get(
        "interpretation"
    )
)

O perfil vocal estimado é compatível com voz feminina. Não foram identificadas palavras-chave clínicas de alerta na transcrição. Os sinais observados vieram principalmente de características acústicas da voz, como variação de pitch, pausas, energia ou intensidade vocal. Como a transcrição não contém termos clínicos de alerta, esses sinais devem ser tratados como evidências complementares de baixa confiança. A intensidade vocal foi classificada como alta. Foram registradas oscilações vocais, que podem indicar variação emocional, mas isoladamente não confirmam ansiedade ou tensão. Foram identificadas pausas ou hesitações na fala, interpretadas apenas como sinal acústico complementar. A análise acústica registrou possível tensão vocal, sem confirmação clínica isolada. A variação do pitch foi elevada, sendo considerada um sinal acústico complementar, mas não deve ser interpretada isoladamente como ansiedade. A proporção de silêncio ou pausas foi elevada, podendo refletir ritmo de fala, gra

### RESULTADO MULTIMODAL

Nesta etapa é exibido o resultado da fusão multimodal entre áudio e vídeo.

In [23]:
pprint(result["multimodal_result"])

{'adjusted_video_score': 0.0,
 'alert': False,
 'audio_risk_level': 'baixo',
 'audio_score': 0.25,
 'display_evidences': ['person_detected',
                       'face_detected',
                       'possible_body_tension'],
 'evidences': ['person_detected',
               'face_detected',
               'possible_body_tension',
               'voice_instability',
               'elevated_voice_tension',
               'speech_hesitation'],
 'final_score': 0.25,
 'fusion_strategy': 'audio_only',
 'interpretation': ['A análise de áudio apresentou baixo nível de risco. Os '
                    'sinais acústicos detectados foram considerados fracos e '
                    'complementares, sem evidência textual clínica de alerta.',
                    'A análise de postura corporal identificou sinais não '
                    'verbais complementares, tratados com baixa ponderação e '
                    'sem valor diagnóstico isolado.',
                    'Foram observados possíveis 

In [24]:
for item in result["multimodal_result"].get(
    "interpretation",
    []
):
    print("-", item)

- A análise de áudio apresentou baixo nível de risco. Os sinais acústicos detectados foram considerados fracos e complementares, sem evidência textual clínica de alerta.
- A análise de postura corporal identificou sinais não verbais complementares, tratados com baixa ponderação e sem valor diagnóstico isolado.
- Foram observados possíveis sinais posturais persistentes de tensão corporal, tratados apenas como evidência complementar e exploratória.


### RESPOSTA FINAL DO AGENTE

O modelo de linguagem gera uma resposta interpretativa utilizando:
- contexto multimodal;
- resultados da fusão;
- informações recuperadas via RAG.

A resposta possui finalidade exclusivamente educacional.

In [25]:
print(result["answer"])

 1. Resumo da avaliação: A análise multimodal não identificou sinais de risco emocional significativos.
2. Evidências observadas: Não foram detectados sinais faciais ou corporais que indicassem emoções fortes ou persistentes.
3. Integração multimodal: Os sinais acústicos, vídeo e postura corporal foram analisados individualmente e como evidências complementares, sem indicar risco emocional relevante isoladamente.
4. Nível de risco: Baixo.
5. Recomendação: A análise não identificou sinais críticos. Recomenda-se manter observação e buscar orientação profissional apenas em caso de agravamento ou persistência dos sintomas.
6. Limitações da análise:
   - A análise multimodal é apenas apoio à triagem clínica.
   - O sistema não realiza diagnóstico médico, psicológico ou psiquiátrico.
   - Expressões faciais indicam apenas emoções aparentes.
   - Sinais de postura corporal indicam apenas padrões não verbais aparentes.
   - Sinais de áudio, vídeo e postura devem ser interpretados como evidênci

### CONCLUSÃO

O teste validou:
- integração multimodal;
- upload e download via Amazon S3;
- processamento de vídeo;
- processamento de áudio;
- análise vocal;
- fusão multimodal;
- geração automática de respostas;
- integração completa do LangGraph.

As análises possuem finalidade educacional e de apoio à triagem preventiva, não representando diagnóstico médico ou psicológico.

### LIMPEZA S3

Após os testes, os arquivos podem ser removidos do S3 para evitar armazenamento desnecessário.

In [26]:
for key in [video_s3_key, audio_s3_key]:

    s3.delete_object(
        Bucket=bucket,
        Key=key
    )

    print("Removido do S3:", key)

Removido do S3: inputs/videos/HappyPregnantWoman.mp4
Removido do S3: inputs/audios/happy.wav


## Interpretação do Resultado Multimodal

Nesta execução, a fusão multimodal classificou o cenário como **baixo risco**, apresentando `final_score = 0.25` e ausência de alerta crítico (`alert = False`).

Apesar de terem sido detectados alguns sinais acústicos, como:
- instabilidade vocal (`voice_instability`);
- tensão vocal elevada (`elevated_voice_tension`);
- hesitações na fala (`speech_hesitation`);

o sistema interpretou esses achados como **sinais complementares fracos**, sem evidências textuais ou comportamentais suficientes para indicar sofrimento emocional relevante.

Sem predominância consistente de emoções negativas. O `video_score = 0.0` reforça que o módulo visual não identificou padrões relevantes de medo, tristeza ou sofrimento.

Esse comportamento é coerente com vídeos contendo:
- expressão neutra;
- felicidade leve;
- fala natural;
- comunicação espontânea.

Mesmo que o vídeo represente um cenário de felicidade, pequenas oscilações acústicas podem ocorrer naturalmente devido a:
- entonação;
- intensidade vocal;
- ritmo da fala;
- qualidade do áudio;
- características individuais da voz.

Por esse motivo, o pipeline foi ajustado para interpretar sinais acústicos isolados de forma cautelosa, reduzindo falsos positivos e evitando inferências emocionais exageradas.

O resultado demonstra que a estratégia de fusão multimodal está funcionando adequadamente ao:
- priorizar evidências consistentes;
- evitar interpretações clínicas indevidas;
- utilizar sinais acústicos apenas como apoio complementar;
- impedir que pequenas variações vocais sejam tratadas como indícios clínicos.

Por fim, o sistema reforça que a análise possui finalidade exclusivamente preventiva e educacional, não realizando diagnóstico médico, psicológico ou psiquiátrico.

## Cenário 2 -> SAD

In [9]:
videoSad = r"C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\video\CryWoman.mp4"

print("Vídeo existe?", os.path.exists(videoSad))
print(videoSad)

Vídeo existe? True
C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\video\CryWoman.mp4


In [10]:
audio_output = Path(
    r"C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\audio\sad.wav"
)
print("Áudio encontrado?", audio_output.exists())
print(audio_output)

Áudio encontrado? True
C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\audio\sad.wav


In [11]:
video_s3_key = (
    "inputs/videos/CryWoman.mp4"
)

audio_s3_key = (
    "inputs/audios/sad.wav"
)

s3.upload_file(
    str(videoSad),
    bucket,
    video_s3_key
)

print("Vídeo enviado para o S3.")

Vídeo enviado para o S3.


In [12]:
s3.upload_file(
    str(audio_output),
    bucket,
    audio_s3_key
)

print("Áudio enviado para o S3.")

Áudio enviado para o S3.


In [13]:
for key in [video_s3_key, audio_s3_key]:

    response = s3.head_object(
        Bucket=bucket,
        Key=key
    )

    print(
        key,
        "-",
        response["ContentLength"],
        "bytes"
    )

inputs/videos/CryWoman.mp4 - 16503020 bytes
inputs/audios/sad.wav - 2268894 bytes


In [17]:
stateSad = {

    "question": (
        "Avalie possíveis sinais de risco "
        "emocional na análise multimodal."
    ),

    "audio_s3_key": audio_s3_key,

    "video_s3_key": video_s3_key,

    "audio_language": "pt-BR",

    "patient_id": "sad_woman"
}

In [18]:
result = app.invoke(stateSad)

print("Pipeline executado com sucesso.")

Pipeline executado com sucesso.


In [20]:
pprint(result["video_result"])

{'flags': ['person_detected',
           'face_detected',
           'sad_expression',
           'disgusted_expression',
           'persistent_sadness'],
 'interpretation': ['Foram detectadas faces no vídeo, permitindo análise de '
                    'expressões emocionais aparentes.',
                    'A distribuição percentual das emoções aparentes '
                    'consideradas na análise foi: SAD: 97.56%, DISGUSTED: '
                    '2.44%.',
                    'Foram observados sinais visuais persistentes associados a '
                    'tristeza aparente.'],
 'limitations': ['A análise facial indica apenas expressões aparentes, não '
                 'estado psicológico real.',
                 'O sistema não realiza diagnóstico médico ou psicológico.',
                 'Os sinais visuais devem ser interpretados apenas como apoio '
                 'à triagem.',
                 'A análise de postura corporal é complementar, exploratória e '
                 '

In [19]:
from src.multimodal.video_processor import analyze_video

video_result = analyze_video(videoSad)

print("Posture score:", video_result.get("posture_score"))
print("Posture flags:", video_result.get("posture_flags"))
print("Frames com pose:", video_result.get("metadata", {}).get("frames_with_pose"))

video_result.get("posture_interpretation")

Posture score: 0.0
Posture flags: ['possible_retracted_posture']
Frames com pose: 8


['Foram observados possíveis sinais persistentes de postura retraída, tratados apenas como evidência complementar e exploratória.']

In [20]:
for item in result["video_result"].get(
    "interpretation",
    []
):
    print("-", item)

- Foram detectadas faces no vídeo, permitindo análise de expressões emocionais aparentes.
- A distribuição percentual das emoções aparentes consideradas na análise foi: SAD: 97.56%, DISGUSTED: 2.44%.
- Foram observados sinais visuais persistentes associados a tristeza aparente.
- Foram observados possíveis sinais persistentes de postura retraída, tratados apenas como evidência complementar e exploratória.


In [21]:
pprint(result["audio_result"])

{'detected_categories': {'depressao_pos_parto_ou_sofrimento_emocional': ['não '
                                                                         'tenho '
                                                                         'vontade']},
 'flags': ['não tenho vontade',
           'voice_instability',
           'elevated_voice_tension',
           'speech_hesitation'],
 'interpretation': 'O perfil vocal estimado é compatível com voz feminina. A '
                   'análise textual identificou categorias associadas a '
                   'depressao_pos_parto_ou_sofrimento_emocional. Os principais '
                   'indicadores textuais e vocais encontrados foram: não tenho '
                   'vontade, voice_instability, elevated_voice_tension, '
                   'speech_hesitation. A intensidade vocal foi classificada '
                   'como alta. Foram registradas oscilações vocais, que podem '
                   'indicar variação emocional, mas isoladamente não co

In [24]:
print(
    result["audio_result"].get(
        "interpretation"
    )
)

O perfil vocal estimado é compatível com voz feminina. A análise textual identificou categorias associadas a depressao_pos_parto_ou_sofrimento_emocional. Os principais indicadores textuais e vocais encontrados foram: não tenho vontade, voice_instability, elevated_voice_tension, speech_hesitation. A intensidade vocal foi classificada como alta. Foram registradas oscilações vocais, que podem indicar variação emocional, mas isoladamente não confirmam ansiedade ou tensão. Foram identificadas pausas ou hesitações na fala, interpretadas apenas como sinal acústico complementar. A análise acústica registrou possível tensão vocal, sem confirmação clínica isolada. A variação do pitch foi elevada, sendo considerada um sinal acústico complementar, mas não deve ser interpretada isoladamente como ansiedade. A proporção de silêncio ou pausas foi elevada, podendo refletir ritmo de fala, gravação, interpretação do ator ou hesitação. Os sinais identificados não representam diagnóstico médico ou psicológ

In [25]:
pprint(result["multimodal_result"])

{'adjusted_video_score': 0.7,
 'alert': True,
 'audio_risk_level': 'alto',
 'audio_score': 0.77,
 'display_evidences': ['person_detected',
                       'face_detected',
                       'sad_expression',
                       'disgusted_expression',
                       'persistent_sadness',
                       'possible_retracted_posture',
                       'não tenho vontade',
                       'voice_instability',
                       'elevated_voice_tension',
                       'speech_hesitation'],
 'evidences': ['person_detected',
               'face_detected',
               'sad_expression',
               'disgusted_expression',
               'persistent_sadness',
               'possible_retracted_posture',
               'não tenho vontade',
               'voice_instability',
               'elevated_voice_tension',
               'speech_hesitation'],
 'final_score': 0.74,
 'fusion_strategy': 'audio_60_video_40_with_posture',
 'inter

In [26]:
for item in result["multimodal_result"].get(
    "interpretation",
    []
):
    print("-", item)

- A análise de áudio apresentou sinais relevantes de possível ansiedade, agitação ou fadiga emocional.
- A análise de vídeo apresentou sinais visuais relevantes de possível desconforto emocional.
- A análise de postura corporal identificou sinais não verbais complementares, tratados com baixa ponderação e sem valor diagnóstico isolado.
- Foram observados possíveis sinais persistentes de postura retraída, tratados apenas como evidência complementar e exploratória.
- Foram observadas expressões aparentes associadas a tristeza.
- Foram observados possíveis sinais de postura retraída, usados apenas como evidência complementar.


In [27]:
print(result["answer"])

 1. Resumo da avaliação: A análise multimodal identificou sinais persistentes de tristeza, possível agitação ou fadiga emocional, e possíveis sinais de postura retraída. O score final foi 0.74, indicando alto nível de risco.
2. Evidências observadas: Foram identificados sinais relevantes de tristeza, possível agitação ou fadiga emocional (sad_expression, persistent_sadness, voice_instability, elevated_voice_tension, speech_hesitation), e possíveis sinais de postura retraída (possible_retracted_posture).
3. Integração multimodal: A análise de áudio apresentou sinais relevantes de possível ansiedade, agitação ou fadiga emocional. A análise de vídeo apresentou sinais visuais relevantes de possível desconforto emocional. A análise de postura corporal identificou sinais não verbais complementares, tratados com baixa ponderação e sem valor diagnóstico isolado.
4. Nível de risco: O nível de risco foi alto (0.74).
5. Recomendação: É recomendada priorizar avaliação por profissional de saúde, es

In [28]:
for key in [video_s3_key, audio_s3_key]:

    s3.delete_object(
        Bucket=bucket,
        Key=key
    )

    print("Removido do S3:", key)

Removido do S3: inputs/videos/CryWoman.mp4
Removido do S3: inputs/audios/sad.wav


### Interpretação da Análise Multimodal — Cenário de Tristeza e Sofrimento Emocional

Nesta execução, o pipeline multimodal classificou o cenário como **alto risco emocional**, combinando evidências provenientes da análise visual e acústica.

No módulo de vídeo, foram detectadas expressões faciais predominantemente associadas a:
- medo (`FEAR = 97.56%`);
- desgosto (`DISGUSTED = 2.44%`);

Além disso, o sistema identificou:
- persistência de emoções negativas ao longo dos frames;
- variações emocionais contínuas;
- presença facial consistente durante a análise.

O `video_score = 0.7` indica forte relevância dos sinais visuais observados, sugerindo um padrão compatível com desconforto emocional aparente.

Na análise de áudio, também foram identificados sinais complementares associados a:
- instabilidade vocal;
- tensão na fala;
- hesitação verbal;
- possíveis sinais de fadiga emocional ou ansiedade.

A combinação dessas evidências elevou o resultado final da fusão multimodal para nível de risco alto.

O comportamento observado é coerente com vídeos contendo:
- expressões de sofrimento;
- medo aparente;
- tensão emocional;
- tristeza persistente;

A estratégia multimodal demonstrou maior robustez ao não depender exclusivamente de um único sinal. Nesse caso:
- o vídeo apresentou forte predominância de emoções negativas;
- o áudio reforçou o contexto emocional detectado visualmente;
- a fusão aumentou a confiança interpretativa do pipeline.

Outro ponto relevante é que o sistema manteve uma abordagem cautelosa e ética:
- não realizou diagnóstico clínico;
- não afirmou condições psicológicas;
- interpretou os sinais apenas como indícios aparentes de sofrimento emocional.

As limitações continuam explícitas no resultado, especialmente porque:
- emoções faciais representam apenas estados aparentes;
- iluminação, qualidade do vídeo e enquadramento podem afetar a análise;
- sinais acústicos são tratados como evidências complementares;
- a confirmação clínica depende de avaliação profissional especializada.

Esse resultado demonstra que a arquitetura multimodal conseguiu identificar coerência entre:
- expressão facial;
- comportamento emocional visual;
- características acústicas da fala.

Consequentemente, o pipeline apresentou maior sensibilidade para cenários de sofrimento emocional mais evidente, reduzindo a probabilidade de falso negativo em situações potencialmente críticas.

## Cenário 3 -> FEAR

In [29]:
videoFear = r"C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\video\FearWoman.mp4"

print("Vídeo existe?", os.path.exists(videoFear))
print(videoFear)

Vídeo existe? True
C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\video\FearWoman.mp4


In [30]:
audio_output = Path(
    r"C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\audio\fear.wav"
)
print("Áudio encontrado?", audio_output.exists())
print(audio_output)

Áudio encontrado? True
C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\audio\fear.wav


In [31]:
video_s3_key = (
    "inputs/videos/FearWoman.mp4"
)

audio_s3_key = (
    "inputs/audios/fear.wav"
)

s3.upload_file(
    str(videoFear),
    bucket,
    video_s3_key
)

print("Vídeo enviado para o S3.")

Vídeo enviado para o S3.


In [32]:
s3.upload_file(
    str(audio_output),
    bucket,
    audio_s3_key
)

print("Áudio enviado para o S3.")

Áudio enviado para o S3.


In [33]:
for key in [video_s3_key, audio_s3_key]:

    response = s3.head_object(
        Bucket=bucket,
        Key=key
    )

    print(
        key,
        "-",
        response["ContentLength"],
        "bytes"
    )

inputs/videos/FearWoman.mp4 - 23119254 bytes
inputs/audios/fear.wav - 2194014 bytes


In [34]:
stateFear = {

    "question": (
        "Avalie possíveis sinais de risco "
        "emocional na análise multimodal."
    ),

    "audio_s3_key": audio_s3_key,

    "video_s3_key": video_s3_key,

    "audio_language": "en-US",

    "patient_id": "fear_woman"
}

In [35]:
result = app.invoke(stateFear)

print("Pipeline executado com sucesso.")

Pipeline executado com sucesso.


In [39]:
pprint(result["video_result"])

{'flags': ['person_detected',
           'face_detected',
           'fear_expression',
           'sad_expression',
           'persistent_fear',
           'emotional_variation_detected'],
 'interpretation': ['Foram detectadas faces no vídeo, permitindo análise de '
                    'expressões emocionais aparentes.',
                    'A distribuição percentual das emoções aparentes '
                    'consideradas na análise foi: SURPRISED: 7.5%, FEAR: '
                    '85.0%, SAD: 7.5%.',
                    'Foram observados sinais visuais persistentes associados a '
                    'medo aparente.',
                    'Foram identificadas variações emocionais ao longo do '
                    'vídeo, usadas como evidência complementar.'],
 'limitations': ['A análise facial indica apenas expressões aparentes, não '
                 'estado psicológico real.',
                 'O sistema não realiza diagnóstico médico ou psicológico.',
                 'Os sinais

In [83]:
for item in result["video_result"].get(
    "interpretation",
    []
):
    print("-", item)

- Foram detectadas faces no vídeo, permitindo análise de expressões emocionais aparentes.
- Foram observados sinais visuais persistentes associados a medo aparente.
- Foram identificadas variações emocionais ao longo do vídeo, usadas como evidência complementar.


In [84]:
pprint(result["audio_result"])

{'detected_categories': {'sinais_de_violencia_ou_medo': ['culpa']},
 'flags': ['culpa',
           'voice_instability',
           'elevated_voice_tension',
           'speech_hesitation'],
 'interpretation': 'O perfil vocal estimado é compatível com voz feminina. A '
                   'análise textual identificou categorias associadas a '
                   'sinais_de_violencia_ou_medo. Os principais indicadores '
                   'textuais e vocais encontrados foram: culpa, '
                   'voice_instability, elevated_voice_tension, '
                   'speech_hesitation. A intensidade vocal foi classificada '
                   'como alta. Foram registradas oscilações vocais, que podem '
                   'indicar variação emocional, mas isoladamente não confirmam '
                   'ansiedade ou tensão. Foram identificadas pausas ou '
                   'hesitações na fala, interpretadas apenas como sinal '
                   'acústico complementar. A análise acústica r

In [40]:
print("Frames com pose:", video_result["metadata"].get("frames_with_pose"))
print("Contagem bruta de postura:", video_result["metadata"].get("posture_raw_counts"))
print("Flags finais de postura:", video_result.get("posture_flags"))
print("Interpretação:", video_result.get("posture_interpretation"))

Frames com pose: 8
Contagem bruta de postura: {'possible_retracted_posture': 8}
Flags finais de postura: ['possible_retracted_posture']
Interpretação: ['Foram observados possíveis sinais persistentes de postura retraída, tratados apenas como evidência complementar e exploratória.']


In [41]:
print(
    result["audio_result"].get(
        "interpretation"
    )
)

O perfil vocal estimado é compatível com voz feminina. Não foram identificadas palavras-chave clínicas de alerta na transcrição. Os sinais observados vieram principalmente de características acústicas da voz, como variação de pitch, pausas, energia ou intensidade vocal. Como a transcrição não contém termos clínicos de alerta, esses sinais devem ser tratados como evidências complementares de baixa confiança. A intensidade vocal foi classificada como alta. Foram registradas oscilações vocais, que podem indicar variação emocional, mas isoladamente não confirmam ansiedade ou tensão. Foram identificadas pausas ou hesitações na fala, interpretadas apenas como sinal acústico complementar. A análise acústica registrou possível tensão vocal, sem confirmação clínica isolada. A variação do pitch foi elevada, sendo considerada um sinal acústico complementar, mas não deve ser interpretada isoladamente como ansiedade. A proporção de silêncio ou pausas foi elevada, podendo refletir ritmo de fala, gra

In [42]:
pprint(result["multimodal_result"])

{'adjusted_video_score': 0.9,
 'alert': True,
 'audio_risk_level': 'baixo',
 'audio_score': 0.25,
 'display_evidences': ['person_detected',
                       'face_detected',
                       'fear_expression',
                       'sad_expression',
                       'persistent_fear',
                       'emotional_variation_detected',
                       'voice_instability',
                       'elevated_voice_tension',
                       'speech_hesitation'],
 'evidences': ['person_detected',
               'face_detected',
               'fear_expression',
               'sad_expression',
               'persistent_fear',
               'emotional_variation_detected',
               'voice_instability',
               'elevated_voice_tension',
               'speech_hesitation'],
 'final_score': 0.51,
 'fusion_strategy': 'audio_60_video_40_with_posture',
 'interpretation': ['A análise de áudio apresentou baixo nível de risco. Os '
                    

In [43]:
print(result["answer"])

 1. Resumo da avaliação: A análise multimodal identificou sinais visuais e acústicos que podem indicar um nível médio de risco emocional.
2. Evidências observadas: Foram detectados sinais de expressões faciais associadas a medo e tristeza, além de sinais acústicos como instabilidade vocal, tensão nas cordas vocais e hesitação na fala.
3. Integração multimodal: A fusão das modalidades apresentou um score final de 0.51, indicando uma maior consistência entre as evidências visuais e acústicas.
4. Nível de risco: O nível de risco identificado foi médio.
5. Recomendação: É recomendada acompanhamento e nova avaliação clínica caso os sinais persistam, se intensifiquem ou estejam associados a sofrimento emocional.
6. Limitações da análise: A análise multimodal é apenas apoio à triagem clínica. O sistema não realiza diagnóstico médico, psicológico ou psiquiátrico. Expressões faciais indicam apenas emoções aparentes. Sinais de postura corporal indicam apenas padrões não verbais aparentes. Sinais

In [44]:
for key in [video_s3_key, audio_s3_key]:

    s3.delete_object(
        Bucket=bucket,
        Key=key
    )

    print("Removido do S3:", key)

Removido do S3: inputs/videos/FearWoman.mp4
Removido do S3: inputs/audios/fear.wav


## Interpretação da Análise Multimodal — Cenário de Medo e Alta Sensibilidade Emocional

Neste cenário, a análise multimodal classificou o conteúdo como **alto risco emocional**, apresentando `risk_score = 0.9`, com predominância de sinais associados a medo, tensão emocional e sofrimento aparente.

A análise visual identificou forte ocorrência da emoção:
- medo (`FEAR = 85%`);

além de pequenas ocorrências de:
- tristeza (`SAD = 7.5%`);
- surpresa/tensão (`SURPRISED = 7.5%`).

Os resultados indicam que as expressões faciais observadas permaneceram consistentes durante a maior parte dos frames analisados, sugerindo persistência de sinais emocionais negativos ao longo do vídeo.

O pipeline também identificou:
- transições emocionais recorrentes;
- presença facial contínua;
- estabilidade na detecção da pessoa em cena;
- múltiplos frames com alta confiança do Rekognition.

Na análise de áudio, foram observados sinais complementares associados a:
- instabilidade vocal;
- tensão elevada na fala;
- hesitação verbal;
- possível agitação emocional.

A combinação entre:
- expressões faciais de medo;
- sinais acústicos de tensão;
- consistência temporal das emoções;

resultou em elevada confiança multimodal para classificação do risco.

O comportamento observado é compatível com cenários contendo:
- medo aparente;
- desconforto emocional intenso;
- insegurança;
- sofrimento emocional visível;
- tensão psicológica perceptível na comunicação.

Um ponto importante deste resultado é que a arquitetura multimodal não se baseou apenas em um frame isolado. A análise considerou:
- amostragem temporal de frames;
- frequência das emoções detectadas;
- persistência emocional;
- combinação entre modalidades.

Isso reduz a chance de classificações ocasionais causadas por expressões momentâneas ou ruídos visuais.

Mesmo com alto nível de risco identificado, o sistema mantém uma abordagem ética e cautelosa:
- não realiza diagnóstico clínico;
- não afirma transtornos psicológicos;
- interpreta apenas emoções aparentes;
- utiliza áudio e vídeo como evidências complementares de apoio preventivo.

Esse resultado demonstra que o pipeline multimodal apresentou elevada sensibilidade para detectar cenários de sofrimento emocional mais evidente, especialmente quando sinais visuais e acústicos convergem de maneira consistente.